In [ ]:
import pandas as pd
import numpy as np

# Load datasets
lc = pd.read_csv('./data/_LC_dataset_v_1_1L.csv')
cr_list = pd.read_csv('./cr/full_new_isins.csv')
unmatched = pd.read_excel('./cr/input/unmatched_bocconi_reports.xlsx')

In [2]:
# Helper function to ensure each cell is a list
def flatten_value(x):
    if isinstance(x, list):
        return x
    elif isinstance(x, str):
        return [code.strip() for code in x.split(',')]
    elif pd.isna(x):
        return []
    else:
        return []

# Apply the function to the column so each value is a list
lc['full_isin_list'] = lc['full_isin_list'].apply(flatten_value)

# Create a single flattened list from the entire column:
flat_list = [item for sublist in lc['full_isin_list'] for item in sublist]

# Merge the flattened list with the ISIN codes from cr_list, ensuring uniqueness
merged_list = list(set(flat_list).union(set(cr_list['ISIN Code'])))

# Remove empty strings
merged_list = [x for x in merged_list if x != '']

In [3]:
unmatched['ISIN'] = unmatched['ISIN'].apply(flatten_value)
unmatched.drop_duplicates(subset=['ISIN'], inplace=True)
unmatched = unmatched[[not any(isin in merged_list for isin in isin_list) for isin_list in unmatched['ISIN']]].reset_index(drop=True)

In [5]:
unmatched.to_csv('./cr/to_check_bocconi.csv', index=False)